In [1]:
!pip install pytorchvideo torchvision av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 51.6 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=e715ce2e54907e39bd25251f0b4fb6830f4f989890d31dc9c24a4e0b53c6f29e
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=68c4e3c9ae888b1d6abe133cf9a51648c64c7be949ef536d62a433468769fc4b
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel f

In [2]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
# Trick pytorchvideo into thinking the old deprecated module still exists
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from google.colab import drive

import pytorchvideo.data
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo

# Import Swin3D from TorchVision
from torchvision.models.video import swin3d_t, Swin3D_T_Weights

# --- 2. CONFIGURATION ---
drive.mount('/content/drive', force_remount=True)

DATA_PATH = "/content/drive/MyDrive/video_data/train"
CLASSES = ['walking', 'running']
BATCH_SIZE = 2
NUM_FRAMES = 32  # Swin3D traditionally expects 32 frames for Kinetics-400
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. ROBUST DATASET DEFINITION ---
labeled_video_paths = []
for i, cls_name in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls_name)
    if not os.path.isdir(cls_path): continue
    for f in os.listdir(cls_path):
        if f.lower().endswith(('.mp4', '.avi', '.mov')):
            labeled_video_paths.append((os.path.join(cls_path, f), {'label': i}))

# Swin3D Transform Pipeline
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0), # Normalize to 0-1
            # Swin uses standard ImageNet normalization values
            NormalizeVideo(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            # Swin3D default crop size is 224x224
            Resize((224, 224))
        ])
    )
])

# Safely extract 2-second clips
clip_sampler = pytorchvideo.data.make_clip_sampler("uniform", 2.0)

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=labeled_video_paths,
    clip_sampler=clip_sampler,
    transform=video_transform,
    decode_audio=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)

# --- 4. MODEL INITIALIZATION ---
print("Downloading Swin3D-Tiny model from TorchVision...")
weights = Swin3D_T_Weights.KINETICS400_V1
model = swin3d_t(weights=weights)

# Replace the classification head for Swin3D
model.head = nn.Linear(model.head.in_features, len(CLASSES))

model = model.to(DEVICE)

# --- 5. TRAINING LOOP ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print("\n--- Starting Swin3D Training ---")
model.train()

for epoch in range(EPOCHS):
    running_loss = 0.0
    batches = 0

    for batch_dict in train_loader:
        # Swin3D expects shape: (Batch, Channels, Time, Height, Width)
        videos = batch_dict['video'].to(DEVICE)
        labels = batch_dict['label'].to(DEVICE)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batches += 1

    avg_loss = running_loss / batches if batches > 0 else 0
    print(f"Epoch [{epoch+1}/{EPOCHS}] Average Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), "video_swin3d_final.pth")
print("Training Complete. Model Saved.")

# --- 6. TARGET FILE / INFERENCE ---
NEW_VIDEO_PATH = "/content/drive/MyDrive/video_data/test/v_PlayingCello_g04_c02.avi"
from pytorchvideo.data.encoded_video import EncodedVideo

def run_guaranteed_inference(video_path):
    model.eval()
    try:
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)

        # Apply the exact same transform
        video_data = video_transform(video_data)

        # Add batch dimension and send to GPU
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        print("-" * 30)
        print(f"FILE: {os.path.basename(video_path)}")
        print(f"PREDICTED: {CLASSES[pred_idx.item()].upper()}")
        print(f"CONFIDENCE: {conf.item()*100:.2f}%")
        print("-" * 30)

    except Exception as e:
        print(f"Inference failed: {e}")

run_guaranteed_inference(NEW_VIDEO_PATH)

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Mounted at /content/drive
Downloading: "https://download.pytorch.org/models/swin3d_t-7615ae03.pth" to /root/.cache/torch/hub/checkpoints/swin3d_t-7615ae03.pth


100%|██████████| 122M/122M [00:00<00:00, 138MB/s]



--- Starting Swin3D Training ---
Epoch [1/3] Average Loss: 0.1753
Epoch [2/3] Average Loss: 0.0063
Epoch [3/3] Average Loss: 0.0000
Training Complete. Model Saved.
------------------------------
FILE: v_PlayingCello_g04_c02.avi
PREDICTED: RUNNING
CONFIDENCE: 100.00%
------------------------------
